In [2]:
%load_ext autoreload
%autoreload 2

In [1]:
from attribscope.classifier.classifier import (
    MLPClassifier,
    train, infer, quick_eval,
    seed_everything
)
from attribscope.svd2.utils import (
    load_representations,
    _resolve_dir,
    split_data,
    compute_metrics,
    get_mistake_meta
)
from attribscope.svd2.computation import (
    fit_one, score_one, fit_all, score_all
)
from attribscope.svd2.utils import run_metrics
import torch
from pathlib import Path
import re
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np


In [3]:
def downsample(X: torch.Tensor, y: torch.Tensor, weight: int = 1):
    """
    Undersample label 0 so that n(label 0) / n(label 1) = weight.
    All label 1 samples are kept.
    """
    idx_pos = (y == 1).nonzero(as_tuple=True)[0]
    idx_neg = (y == 0).nonzero(as_tuple=True)[0]

    n_neg_keep = min(len(idx_pos) * weight, len(idx_neg))
    idx_neg_sampled = idx_neg[torch.randperm(len(idx_neg))[:n_neg_keep]]

    idx_all = torch.cat([idx_pos, idx_neg_sampled])
    idx_all = idx_all[torch.randperm(len(idx_all))]  # shuffle

    return X[idx_all], y[idx_all]


def upsample(X: torch.Tensor, y: torch.Tensor, weight: int = 1):
    """
    Upsample label 1 (with replacement) so that n(label 0) / n(label 1) = weight.
    All label 0 samples are kept. If the current ratio is already <= weight,
    positives are left untouched (no downsampling of label 1).
    """
    idx_pos = (y == 1).nonzero(as_tuple=True)[0]
    idx_neg = (y == 0).nonzero(as_tuple=True)[0]

    n_pos_target = max(len(idx_neg) // weight, len(idx_pos))
    idx_pos_sampled = idx_pos[torch.randint(0, len(idx_pos), (n_pos_target,))]

    idx_all = torch.cat([idx_pos_sampled, idx_neg])
    idx_all = idx_all[torch.randperm(len(idx_all))]  # shuffle

    return X[idx_all], y[idx_all]

def key_hidden(s):
    if s == 'embed': return (-1, 0, '')
    match = re.search(r'(\d+)', s)
    num = int(match.group(1))
    return (0, num, s)

def key_grads(s):
    if s == 'embed': return ['', -1]
    parts = re.split(r'(\d+)', s)
    return [int(p) if p.isdigit() else p for p in parts]

In [4]:
REPS_ROOT:    Path = Path("/data/hoang/attrib/outputs")
DATA_ROOT:    Path = Path("data/ww")
OUTPUTS_ROOT: Path = None

MODEL:        str   = ["llama-3.1-8b", "qwen3-8b"][0] 
SUBSET:       str   = ["algorithm-generated", "hand-crafted"][0] 
REP_TYPE:     str   = ["grads", "hidden"][1]  
POOLING:      str   = ["grad", "mean", "last"][1]   # grads -> grad, hidden -> last | mean
WEIGHT_NAMES: str | list[str] = "all"
LOSS:         str   = "ntp"   
TEMPERATURE:  float | None = None

# RATIO:        float = 0.5
SEED:         int   = 100
DEVICE: torch.device = torch.device("cuda:5")

In [5]:
rep_dir = _resolve_dir(
    root_dir=REPS_ROOT, 
    model=MODEL, 
    subset=SUBSET,
    rep_type=REP_TYPE, 
    loss=LOSS, 
    temperature=TEMPERATURE,
    dir_type="representations"
)
data_dir = DATA_ROOT / SUBSET

print(f"Representation dir: {rep_dir}")
print(f"Data dir:           {data_dir}")

files = sorted(rep_dir.glob("*.safetensors"), key=lambda x: int(x.stem))
assert files, (f"No .safetensors files in {rep_dir}")

train_files, test_files = split_data(files, 0.5, SEED)
train_files, val_files  = split_data(train_files, 0.8, SEED)

print(f"Total train trajectories: {len(train_files)}")
print(f"Total val trajectories:   {len(val_files)}")
print(f"Total test trajectories:  {len(test_files)}")

Representation dir: /data/hoang/attrib/outputs/hidden/llama-3.1-8b/reps/algorithm-generated
Data dir:           data/ww/algorithm-generated
Total train trajectories: 50
Total val trajectories:   13
Total test trajectories:  63


In [6]:
rep_kwargs = dict(
    rep_dir=rep_dir,
    data_dir=data_dir,
    pooling=POOLING,
    weight_names=WEIGHT_NAMES,
    device=DEVICE,
)

train_reps = load_representations(**rep_kwargs, files=train_files)
val_reps   = load_representations(**rep_kwargs, files=val_files)
test_reps  = load_representations(**rep_kwargs, files=test_files)

Loading representations:   0%|          | 0/50 [00:00<?, ?it/s]

Loading representations: 100%|██████████| 63/63 [00:01<00:00, 55.19it/s]


In [8]:
sort_key = dict(grads=key_grads, hidden=key_hidden)[REP_TYPE]
layer_idxs = list(train_reps.stores.keys())
layer_idxs = sorted(layer_idxs, key=sort_key)

LAYER_IDX  = layer_idxs[23]
print(f"Layers: {layer_idxs}\n")
print(f"Selected layer: {LAYER_IDX}")

Layers: ['embed', 'act/0', 'act/1', 'act/2', 'act/3', 'act/4', 'act/5', 'act/6', 'act/7', 'act/8', 'act/9', 'act/10', 'act/11', 'act/12', 'act/13', 'act/14', 'act/15', 'act/16', 'act/17', 'act/18', 'act/19', 'act/20', 'act/21', 'act/22', 'act/23', 'act/24', 'act/25', 'act/26', 'act/27', 'act/28', 'act/29', 'act/30', 'act/31', 'act/31_normed']

Selected layer: act/22


In [9]:
len(layer_idxs)

34

In [10]:
def run_one(
    train_reps,
    val_reps,
    test_reps,
    layer_idx,
    threshold,
    device
):
    # --- TODO
    # Convert this into a sweep file
    # Add SVD-based scoring for comparison.
    # Also add grad
    
    # --- Fitting SVD components
    svd = fit_all(train_reps.stores, n_components=10)

    # --- Getting validation scores
    n_components_score = list(range(1, 11))
    score_kwargs = dict(svd=svd, n_components_score=n_components_score, device=device)
    train_scores = score_all(train_reps.stores, **score_kwargs)
    val_scores   = score_all(val_reps.stores, **score_kwargs)

    # --- Calculate threshold score for wild training data
    val_metrics = run_metrics(val_scores, keeper=val_reps.keeper, ks=[1])
    config = val_metrics.query(
        f"weight == '{layer_idx}' and direction == 'asc'"
    ).sort_values(["step_acc"], ascending=False).iloc[0].to_dict()
    QUERY = (
        f"weight == '{layer_idx}' "
        f"and pooling == '{config["pooling"]}' "
        f"and method  == '{config["method"]}' "
        f"and c == {config["c"]} "
        f"and centered == {config["centered"]}"
    )
    print(f"Using best validation config:\n{QUERY}")
    print(f"The best validation results with direct projection on SVD components")
    print(f"Step@1: {config['step_acc']:.4f} Agent@1: {config['agent_acc']:.4f}\n")
    pseudo_scores = pd.DataFrame(train_scores).query(QUERY).iloc[0].scores
    wild_threshold = np.sort(pseudo_scores)[int(len(pseudo_scores) * threshold)]

    # --- Load training data and create pseudo labels
    X_train = train_reps.stores[layer_idx].R
    X_train = X_train.float().to(device)

    y_train = torch.Tensor(
        [idx.is_mistake for idx in  train_reps.keeper.index],
    ).to(device=X_train.device)
    y_train_pseudo = torch.Tensor( # <- use this instead of y_train
        (pseudo_scores < wild_threshold)
    ).to(device=X_train.device)

    # --- Load validation data
    X_val = val_reps.stores[layer_idx].R
    X_val = X_val.float().to(device)
    y_val = torch.Tensor(
        [idx.is_mistake for idx in  val_reps.keeper.index],
    ).to(device=X_val.device)
    
    # --- Load test data
    X_test = test_reps.stores[layer_idx].R
    y_test = torch.Tensor(
        [idx.is_mistake for idx in  test_reps.keeper.index],
    ).to(device=X_test.device)

    train_loader = DataLoader(TensorDataset(X_train, y_train_pseudo), batch_size=512, shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=512, shuffle=False)

    # --- Train with pseudo labels
    seed_everything()
    model = MLPClassifier(
        input_dim=X_train.shape[1], 
        hidden_dim=1024
    )
    clf, metrics = train(
        model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=500,
        learning_rate=0.05,
        weight_decay=3e-4,
        momentum=0.9,
        pos_weight=None,
        logging_steps=100,
        val_metric="f1",
        device=next(iter(train_loader))[0].device,    
    )

    # --- Precompute mistake meta (same for all layers)
    train_mistake_indices, train_mistake_roles = get_mistake_meta(train_reps.keeper)
    val_mistake_indices, val_mistake_roles     = get_mistake_meta(val_reps.keeper)
    test_mistake_indices, test_mistake_roles   = get_mistake_meta(test_reps.keeper)

    # --- Validation metrics
    val_scores = infer(clf, X_val, return_logits=False, device=DEVICE)
    test_metrics = compute_metrics(
        scores=val_scores, keeper=val_reps.keeper,
        mistake_indices=val_mistake_indices, mistake_roles=val_mistake_roles,
        ks=[1], direction="desc",
    )
    val_step_acc, val_agent_acc = list(test_metrics.values())

    # --- Test metrics
    test_scores = infer(clf, X_test, return_logits=False, device=DEVICE)
    test_metrics = compute_metrics(
        scores=test_scores, keeper=test_reps.keeper,
        mistake_indices=test_mistake_indices, mistake_roles=test_mistake_roles,
        ks=[1], direction="desc",
    )
    test_step_acc, test_agent_acc = list(test_metrics.values())

    print(
        f"  Layer {layer_idx:>10} | "
        f"Validation Step@1: {val_step_acc:.4f}  Agent@1: {val_agent_acc:.4f} | "
        f"Test  Step@1: {test_step_acc:.4f}  Agent@1: {test_agent_acc:.4f}"
    )
    final_metrics = dict(
        layer_idx=layer_idx,
        threshold=threshold,
        val_step_acc=val_step_acc,
        val_agent_acc=val_agent_acc,
        test_step_acc=test_step_acc,
        test_agent_acc=test_agent_acc
    )
    return clf, final_metrics

In [12]:
clf, metrics = run_one(
    train_reps = train_reps,
    val_reps   = val_reps,
    test_reps  = test_reps,
    threshold  = 0.05,
    layer_idx  = "act/20",
    device     = DEVICE
)

SVD fit: 100%|██████████| 34/34 [00:00<00:00, 84.12it/s]


Using best validation config:
weight == 'act/20' and pooling == 'mean' and method  == 'proj' and c == 9 and centered == True
The best validation results with direct projection on SVD components
Step@1: 0.4615 Agent@1: 0.6923

Epoch 100/500 - loss: 0.1026 | accuracy: 0.9521 | precision: 0.0000 | recall: 0.0000 | F1: 0.0000 | auroc: 0.5910 | val f1: 0.0000 (best: 0.0000)
Epoch 200/500 - loss: 0.0471 | accuracy: 0.9155 | precision: 0.0000 | recall: 0.0000 | F1: 0.0000 | auroc: 0.4109 | val f1: 0.0000 (best: 0.0000)
Epoch 300/500 - loss: 0.0285 | accuracy: 0.9087 | precision: 0.0000 | recall: 0.0000 | F1: 0.0000 | auroc: 0.5037 | val f1: 0.0000 (best: 0.0000)
Epoch 400/500 - loss: 0.0232 | accuracy: 0.9110 | precision: 0.0500 | recall: 0.0476 | F1: 0.0488 | auroc: 0.4573 | val f1: 0.0000 (best: 0.0000)
Epoch 500/500 - loss: 0.0231 | accuracy: 0.9110 | precision: 0.0500 | recall: 0.0476 | F1: 0.0488 | auroc: 0.3292 | val f1: 0.0000 (best: 0.0000)
Load from best state from epoch 500
Best f1:

In [13]:
metrics

{'layer_idx': 'act/20',
 'threshold': 0.05,
 'val_step_acc': 0.3076923076923077,
 'val_agent_acc': 0.5384615384615384,
 'test_step_acc': 0.4444444444444444,
 'test_agent_acc': 0.6190476190476191}

In [13]:
svd = fit_all(train_reps.stores, n_components=10)

SVD fit:   0%|          | 0/34 [00:00<?, ?it/s]

SVD fit: 100%|██████████| 34/34 [00:00<00:00, 93.54it/s]


In [14]:
n_components_score = list(range(1, 11))
score_kwargs = dict(svd=svd, n_components_score=n_components_score, device=DEVICE)
train_scores = score_all(train_reps.stores, **score_kwargs)
val_scores   = score_all(val_reps.stores, **score_kwargs)

In [19]:
val_metrics = run_metrics(val_scores, keeper=val_reps.keeper, ks=[1])
config = val_metrics.query(
    f"weight == '{LAYER_IDX}' and direction == 'asc'"
).sort_values(["step_acc"], ascending=False).iloc[0].to_dict()
QUERY = (
    f"weight == '{LAYER_IDX}' "
    f"and pooling == '{config["pooling"]}' "
    f"and method  == '{config["method"]}' "
    f"and c == {config["c"]} "
    f"and centered == {config["centered"]}"
)
pseudo_scores = pd.DataFrame(train_scores).query(QUERY).iloc[0].scores

THRESHOLD = [x / 20 for x in range(1, 20)][2] # 0.1
print(f"Threshold: {THRESHOLD}")
wild_threshold = np.sort(pseudo_scores)[int(len(pseudo_scores) * THRESHOLD)]

Threshold: 0.15


In [20]:

# --- Load training data and create pseudo labels
X_train = train_reps.stores[LAYER_IDX].R
y_train = torch.Tensor(
    [idx.is_mistake for idx in  train_reps.keeper.index],
).to(device=X_train.device)
X_train = X_train.float().to(DEVICE)
y_train_pseudo = torch.Tensor(
    (pseudo_scores < wild_threshold)
).to(device=X_train.device)

# --- Load validation data
X_val = val_reps.stores[LAYER_IDX].R
y_val = torch.Tensor(
    [idx.is_mistake for idx in  val_reps.keeper.index],
).to(device=X_val.device)
X_val = X_val.float().to(DEVICE)

# --- Load test data
X_test = test_reps.stores[LAYER_IDX].R
y_test = torch.Tensor(
    [idx.is_mistake for idx in  test_reps.keeper.index],
).to(device=X_test.device)

# X_val, y_val = downsample(X_val, y_val, weight=1)
train_loader = DataLoader(TensorDataset(X_train, y_train_pseudo), batch_size=512, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=512, shuffle=False)

In [21]:
seed_everything()
model = MLPClassifier(
    input_dim=X_train.shape[1], 
    hidden_dim=1024
)
clf, metrics = train(
    model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=300,
    learning_rate=0.05,
    weight_decay=3e-4,
    momentum=0.9,
    pos_weight=None,
    logging_steps=100,
    val_metric="f1",
    device=next(iter(train_loader))[0].device,    
)

Epoch 100/300 - loss: 0.0837 | accuracy: 0.7619 | precision: 0.1803 | recall: 0.1667 | F1: 0.1732 | auroc: 0.5003 | val f1: 0.3478 (best: 0.3478)
Epoch 200/300 - loss: 0.0292 | accuracy: 0.7415 | precision: 0.1364 | recall: 0.1364 | F1: 0.1364 | auroc: 0.5284 | val f1: 0.3200 (best: 0.3478)
Epoch 300/300 - loss: 0.0250 | accuracy: 0.7551 | precision: 0.1818 | recall: 0.1818 | F1: 0.1818 | auroc: 0.5562 | val f1: 0.3200 (best: 0.3478)
Load from best state from epoch 100
Best f1: 0.34782608695652173


In [23]:
test_scores = infer(
    clf, X_test,
    return_logits=False,
    device=X_test.device
)

mistake_indices, mistake_roles = get_mistake_meta(test_reps.keeper)
metrics = compute_metrics(
    scores=test_scores,
    keeper=test_reps.keeper,
    mistake_indices=mistake_indices,
    mistake_roles=mistake_roles,
    ks=[1],
    direction="desc"
)
step_acc, agent_acc = list(metrics.values())
print("Metrics on test set:")
print(f"Step@1: {step_acc:.4f} | Agent@1: {agent_acc:.4f}")

Metrics on test set:
Step@1: 0.3651 | Agent@1: 0.5556


## Run all positions

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from attribscope.classifier.run_all_positions import (
    precompute_scores, get_pseudo_labels,
    prepare_data, get_metrics, run_one,
    key_grads, key_hidden, build_rep_configs
)

from attribscope.svd2.utils import (
    RepresentationStores,
    load_representations,
    _resolve_dir,
    split_data,
    compute_metrics,
    get_mistake_meta
)
from attribscope.classifier.classifier import (
    MLPClassifier, seed_everything
)
import torch
from pathlib import Path

In [6]:
rep_configs = build_rep_configs()
for rep_type, pooling, loss, temperature in rep_configs:
    rep_dir = _resolve_dir(
        root_dir=REPS_ROOT, 
        model=MODEL, 
        subset=SUBSET,
        rep_type=rep_type, 
        loss=loss, 
        temperature=temperature,
        dir_type="representations"
    )
    data_dir = DATA_ROOT / SUBSET

    print(f"Representation dir: {rep_dir}")
    print(f"Data dir:           {data_dir}")

    files = sorted(rep_dir.glob("*.safetensors"), key=lambda x: int(x.stem))
    assert files, (f"No .safetensors files in {rep_dir}")

    train_files, test_files = split_data(files, 0.5, SEED)
    train_files, val_files  = split_data(train_files, 0.8, SEED)

    print(f"Total train trajectories: {len(train_files)}")
    print(f"Total val trajectories:   {len(val_files)}")
    print(f"Total test trajectories:  {len(test_files)}")

Representation dir: /data/hoang/attrib/outputs/grads/llama-3.1-8b/reps/algorithm-generated
Data dir:           data/ww/algorithm-generated
Total train trajectories: 50
Total val trajectories:   13
Total test trajectories:  63
Representation dir: /data/hoang/attrib/outputs/grads/llama-3.1-8b-kl/uniform/reps/algorithm-generated
Data dir:           data/ww/algorithm-generated
Total train trajectories: 50
Total val trajectories:   13
Total test trajectories:  63
Representation dir: /data/hoang/attrib/outputs/grads/llama-3.1-8b-kl/temp_1.2/reps/algorithm-generated
Data dir:           data/ww/algorithm-generated
Total train trajectories: 50
Total val trajectories:   13
Total test trajectories:  63
Representation dir: /data/hoang/attrib/outputs/grads/llama-3.1-8b-kl/temp_1.4/reps/algorithm-generated
Data dir:           data/ww/algorithm-generated
Total train trajectories: 50
Total val trajectories:   13
Total test trajectories:  63
Representation dir: /data/hoang/attrib/outputs/grads/llama-3.

In [5]:
REPS_ROOT:    Path = Path("/data/hoang/attrib/outputs")
DATA_ROOT:    Path = Path("data/ww")
OUTPUTS_ROOT: Path = None

MODEL:        str   = ["llama-3.1-8b", "qwen3-8b"][0] 
SUBSET:       str   = ["algorithm-generated", "hand-crafted"][0] 
REP_TYPE:     str   = ["grads", "hidden"][1]  
POOLING:      str   = ["grad", "mean", "last"][1]   # grads -> grad, hidden -> last | mean
WEIGHT_NAMES: str | list[str] = "all"
LOSS:         str   = "ntp"   
TEMPERATURE:  float | None = None

# RATIO:        float = 0.5
SEED:         int   = 100
DEVICE: torch.device = torch.device("cuda:7")

In [16]:
rep_dir = _resolve_dir(
    root_dir=REPS_ROOT, 
    model=MODEL, 
    subset=SUBSET,
    rep_type=REP_TYPE, 
    loss=LOSS, 
    temperature=TEMPERATURE,
    dir_type="representations"
)
data_dir = DATA_ROOT / SUBSET

print(f"Representation dir: {rep_dir}")
print(f"Data dir:           {data_dir}")

files = sorted(rep_dir.glob("*.safetensors"), key=lambda x: int(x.stem))
assert files, (f"No .safetensors files in {rep_dir}")

train_files, test_files = split_data(files, 0.5, SEED)
train_files, val_files  = split_data(train_files, 0.8, SEED)

print(f"Total train trajectories: {len(train_files)}")
print(f"Total val trajectories:   {len(val_files)}")
print(f"Total test trajectories:  {len(test_files)}")

Representation dir: /data/hoang/attrib/outputs/hidden/llama-3.1-8b/reps/algorithm-generated
Data dir:           data/ww/algorithm-generated
Total train trajectories: 50
Total val trajectories:   13
Total test trajectories:  63


In [17]:
rep_kwargs = dict(
    rep_dir=rep_dir,
    data_dir=data_dir,
    pooling=POOLING,
    weight_names=WEIGHT_NAMES,
    device=DEVICE,
)

train_reps = load_representations(**rep_kwargs, files=train_files)
val_reps   = load_representations(**rep_kwargs, files=val_files)
test_reps  = load_representations(**rep_kwargs, files=test_files)

sort_key = dict(grads=key_grads, hidden=key_hidden)[REP_TYPE]
layer_idxs = list(train_reps.stores.keys())
layer_idxs = sorted(layer_idxs, key=sort_key)
print(f"Layers: {layer_idxs}\n")

train_scores, val_scores = precompute_scores(
    train_reps, val_reps, n_components=10, device=DEVICE
)

Loading representations: 100%|██████████| 63/63 [00:01<00:00, 52.53it/s]


Layers: ['embed', 'act/0', 'act/1', 'act/2', 'act/3', 'act/4', 'act/5', 'act/6', 'act/7', 'act/8', 'act/9', 'act/10', 'act/11', 'act/12', 'act/13', 'act/14', 'act/15', 'act/16', 'act/17', 'act/18', 'act/19', 'act/20', 'act/21', 'act/22', 'act/23', 'act/24', 'act/25', 'act/26', 'act/27', 'act/28', 'act/29', 'act/30', 'act/31', 'act/31_normed']



SVD fit: 100%|██████████| 34/34 [00:00<00:00, 39.22it/s]


In [18]:
LAYER_IDX = "act/20"
THRESHOLD = 0.05

train_loader, val_loader, train_split, val_split, test_split = prepare_data(
    train_reps, val_reps, test_reps, train_scores, val_scores,
    layer_idx=LAYER_IDX, threshold=THRESHOLD, device=DEVICE
).values()

X_train, y_train = train_split
X_val, y_val = val_split
X_test, y_test = test_split

seed_everything(SEED)
model = MLPClassifier(
    input_dim=X_train.shape[1], 
    hidden_dim=1024
)
clf, metrics = run_one(
    model, train_loader, val_loader,
    val_reps, test_reps,
    LAYER_IDX, THRESHOLD,
    epochs = 500,
    logging_steps=50,
    device = DEVICE
)


Using best validation config:
weight == 'act/20' and pooling == 'mean' and method  == 'proj' and c == 9 and centered == True
The best validation results with direct projection on SVD components
Step@1: 0.4615 Agent@1: 0.6923

Epoch 50/500 - loss: 0.1526 | accuracy: 0.9521 | precision: 0.0000 | recall: 0.0000 | F1: 0.0000 | auroc: 0.5332 | val f1: 0.0000 (best: 0.0000)
Epoch 100/500 - loss: 0.1026 | accuracy: 0.9521 | precision: 0.0000 | recall: 0.0000 | F1: 0.0000 | auroc: 0.5928 | val f1: 0.0000 (best: 0.0000)
Epoch 150/500 - loss: 0.0686 | accuracy: 0.9224 | precision: 0.0000 | recall: 0.0000 | F1: 0.0000 | auroc: 0.5262 | val f1: 0.0000 (best: 0.0000)
Epoch 200/500 - loss: 0.0473 | accuracy: 0.9201 | precision: 0.0625 | recall: 0.0476 | F1: 0.0541 | auroc: 0.5703 | val f1: 0.0000 (best: 0.0000)
Epoch 250/500 - loss: 0.0347 | accuracy: 0.9178 | precision: 0.1053 | recall: 0.0952 | F1: 0.1000 | auroc: 0.6131 | val f1: 0.0000 (best: 0.0000)
Epoch 300/500 - loss: 0.0283 | accuracy: 0.90

In [7]:
metrics

NameError: name 'metrics' is not defined

In [6]:
import random
trials = [
    dict(dim=dim, lr=lr, layer_idx=layer_idx)
    for dim in [256, 512, 1024]
    for lr in [5e-3, 1e-2, 2e-2, 3e-2, 4e-2, 5e-2]
    # for lr in [5e-2]
    for layer_idx in ["act/25", "act/26", "act/27", "act/28", "act/29"]
]
random.shuffle(trials)
  # cap budget

In [7]:
len(trials)

90

In [22]:
all_metrics = []
for i, trial in enumerate(trials):
    LAYER_IDX = trial['layer_idx']
    THRESHOLD = 0.05

    train_loader, val_loader, train_split, val_split, test_split = prepare_data(
        train_reps, val_reps, test_reps, train_scores, val_scores,
        layer_idx=LAYER_IDX, threshold=THRESHOLD, device=DEVICE
    ).values()

    X_train, y_train = train_split
    X_val, y_val = val_split
    X_test, y_test = test_split
    
    seed_everything(SEED + i)
    model = MLPClassifier(
        input_dim=X_train.shape[1], 
        hidden_dim=trial['dim']
    )
    clf, metrics = run_one(
        model, train_loader, val_loader,
        val_reps, test_reps,
        trial['layer_idx'], THRESHOLD,
        epochs = 300,
        learning_rate=trial['lr'],
        logging_steps=None,
        device = DEVICE
    )
    all_metrics.append({**trial, **metrics})

Using best validation config:
weight == 'act/26' and pooling == 'mean' and method  == 'proj' and c == 9 and centered == True
The best validation results with direct projection on SVD components
Step@1: 0.3846 Agent@1: 0.5385

Load from best state from epoch 300
Best f1: 0.0
  Layer     act/26 | Validation Step@1: 0.6154  Agent@1: 0.6923 | Test  Step@1: 0.3492  Agent@1: 0.5079
Using best validation config:
weight == 'act/28' and pooling == 'mean' and method  == 'proj' and c == 7 and centered == True
The best validation results with direct projection on SVD components
Step@1: 0.3846 Agent@1: 0.4615

Load from best state from epoch 300
Best f1: 0.0
  Layer     act/28 | Validation Step@1: 0.5385  Agent@1: 0.6154 | Test  Step@1: 0.4127  Agent@1: 0.5397
Using best validation config:
weight == 'act/26' and pooling == 'mean' and method  == 'proj' and c == 9 and centered == True
The best validation results with direct projection on SVD components
Step@1: 0.3846 Agent@1: 0.5385

Load from best s

In [9]:
import pandas as pd
df = pd.DataFrame(all_metrics).sort_values("test_step_acc", ascending=False)

In [19]:
dim_scores = (
    df.groupby("dim")["test_step_acc"]
    .mean()
    .sort_values(ascending=False)
)
best_dim = int(dim_scores.idxmax())
print(f"Best dim: {best_dim}")
print(dim_scores.to_string(), "\n")

Best dim: 1024
dim
1024    0.316931
512     0.311111
256     0.303704 



In [20]:
df_dim = df[df["dim"] == best_dim]
lr_scores = (
    df_dim.groupby("lr")["test_step_acc"]
    .mean()
    .sort_values(ascending=False)
)
best_lr = float(lr_scores.idxmax())
print(f"Best lr: {best_lr}")
print(lr_scores.to_string(), "\n")

Best lr: 0.01
lr
0.010    0.355556
0.020    0.342857
0.050    0.330159
0.030    0.326984
0.040    0.320635
0.005    0.225397 



In [21]:
df_dim

,dim,lr,layer_idx,threshold,val_step_acc,val_agent_acc,test_step_acc,test_agent_acc
5,1024,0.050,act/28,0.05,0.230769,0.461538,0.412698,0.587302
35,1024,0.030,act/28,0.05,0.230769,0.461538,0.412698,0.571429
81,1024,0.040,act/28,0.05,0.230769,0.461538,0.396825,0.571429
70,1024,0.020,act/28,0.05,0.307692,0.538462,0.396825,0.555556
43,1024,0.010,act/28,0.05,0.461538,0.615385,0.380952,0.539683
69,1024,0.010,act/25,0.05,0.538462,0.615385,0.380952,0.523810
10,1024,0.020,act/25,0.05,0.307692,0.538462,0.365079,0.555556
46,1024,0.050,act/27,0.05,0.230769,0.461538,0.349206,0.539683
0,1024,0.010,act/26,0.05,0.615385,0.692308,0.349206,0.507937
82,1024,0.010,act/27,0.05,0.615385,0.692308,0.349206,0.539683


In [ ]:
seed_everything(SEED)
model = MLPClassifier(
    input_dim=X_train.shape[1], 
    hidden_dim=1024
)
clf, metrics = run_one(
    model, train_loader, val_loader,
    val_reps, test_reps,
    LAYER_IDX, THRESHOLD,
    epochs = 300,
    logging_steps=None,
    device = DEVICE
)

{'layer_idx': 'act/27',
 'threshold': 0.05,
 'val_step_acc': 0.23076923076923078,
 'val_agent_acc': 0.46153846153846156,
 'test_step_acc': 0.3333333333333333,
 'test_agent_acc': 0.5238095238095238}